In [ ]:
import pandas as pd
from aloud_database.aloud_database import Database
from utilitary.formatar_telefone import formatar_telefone

db = Database()

In [ ]:
alunos = pd.read_csv('data/bf25_inscritos/bf25_alunos_inscritos.csv')

In [ ]:
survey_1 = pd.read_csv('data/bases_antigas/l08_survey.csv')

def clean_data(survey_1):
    # Corrige nomes de colunas
    survey_1 = survey_1.rename(
        columns={
            'Qual a sua renda mensal? (Caso não tenha renda fixa, você pode selecionar uma média aproximada). O inglês traz muitas oportunidades de crescimento na carreira e é importante reconhecer o seu ponto de partida. ': 'incomme',
            'Qual sua faixa etária?': 'age_range',
            'Email': 'email'
        }
    )
    # Remove colunas desnecessárias
    survey_1 = survey_1.drop(
        columns=[
            'Qual o seu gênero?',
            'Em que país você mora?',
            'Qual sua ocupação atual? ',
            ' Qual o seu maior desejo hoje em relação a fluência em inglês?',
            ' Qual a sua maior dificuldade hoje para alcançar a fluência em inglês?',
            'Qual é o maior problema que você enfrenta por ainda não ser fluente em inglês? (exemplo: perder uma promoção no trabalho, não conseguir viajar para fora sem depender de alguém). ',
            'Qual o seu nível de conhecimento sobre o inglês?',
            'Caso tenha respondido "sim",  descreva um pouco sobre a sua experiência, dificuldades, pontos positivos e negativos, e o quanto gastou com esse curso.',
            'O que você gostaria de aprender no Fluência Destravada?\n',
            'Qual é a sua maior motivação para querer se tornar fluente em inglês? \n',
            'Utm source',
            'Utm Medium',
            'Utm campaign',
            'utm term',
            'utm content',
            'Página cadastro',
            'Score'
        ]
    )
    # Remove coluna de curso de inglês já feito
    survey_1 = survey_1.drop(columns=['Você já fez algum curso de inglês?'])

    # Remove linhas com dados ausentes nas principais colunas
    survey_1 = survey_1.dropna(subset=['email', 'age_range', 'incomme'])

    # Remove duplicatas por email
    survey_1 = survey_1.drop_duplicates(subset=['email'])

    # Padroniza e limpa campos de texto
    survey_1['email'] = survey_1['email'].str.lower().str.strip()
    survey_1['age_range'] = survey_1['age_range'].str.strip()
    survey_1['incomme'] = survey_1['incomme'].str.strip()
    return survey_1

survey_1 = clean_data(survey_1.copy())

In [ ]:
survey_2 = pd.read_csv('data/bases_antigas/l09_survey.csv')

def clean_data(survey_2):
    # Renomeia a coluna 'monthly_income' para 'incomme'
    if 'monthly_income' in survey_2.columns:
        survey_2 = survey_2.rename(columns={'monthly_income': 'incomme'})
    # Remove colunas desnecessárias se existirem
    cols_to_drop = [
        'biggest_fluency_desire', 'biggest_fluency_difficulty', 'biggest_fluency_problem', 'english_proficiency_level', 
        'has_taken_english_course', 'english_course_experience', 'currently_studying_english', 
        'learn_in_fluency_unlocked', 'fluency_motivation', 'in_whatsapp_group', 'score', 'live_session_topics', 
        'gender_points', 'age_range_points', 'current_occupation_points', 'monthly_income_points', 
        'biggest_fluency_desire_points', 'biggest_fluency_difficulty_points', 'english_proficiency_level_points', 
        'has_taken_english_course_points', 'row', 'country_of_residence', 'current_occupation', 
        'lgpd_consent', 'id', 'lead_id', 'token', 'gender', 'active_id'
    ]
    cols_to_drop = [col for col in cols_to_drop if col in survey_2.columns]
    survey_2 = survey_2.drop(columns=cols_to_drop)
    
    # Garante o tipo datetime para 'submitted_at' se existir
    if 'submitted_at' in survey_2.columns:
        survey_2['submitted_at'] = pd.to_datetime(survey_2['submitted_at'], errors='coerce')
    
    # Remove linhas com dados ausentes nas principais colunas
    for col in ['email', 'age_range', 'incomme']:
        if col not in survey_2.columns:
            raise RuntimeError(f"Coluna obrigatória ausente: {col}")
    survey_2 = survey_2.dropna(subset=['email', 'age_range', 'incomme'])

    # Padroniza e limpa campos de texto
    survey_2['email'] = survey_2['email'].astype(str).str.lower().str.strip()
    survey_2['age_range'] = survey_2['age_range'].astype(str).str.strip()
    survey_2['incomme'] = survey_2['incomme'].astype(str).str.strip()

    # Remove duplicatas por email, se desejar igual ao survey_1
    survey_2 = survey_2.drop_duplicates(subset=['email'])

    return survey_2

survey_2 = clean_data(survey_2.copy())

In [ ]:
survey_3 = pd.read_csv('data/bases_antigas/all_surveys.csv')
survey_3 = survey_3.dropna(subset=['email'])


In [ ]:
survey_concat = pd.concat([survey_1, survey_2, survey_3], ignore_index=True)
survey_concat = survey_concat.drop_duplicates(subset=['email'])

In [ ]:
leads_1 = pd.read_csv('data/bases_antigas/l08_leads.csv')

import re

def clean_data(leads_1):
    # Rename column 'Nome' to 'name'
    leads_1 = leads_1.rename(columns={'Nome': 'name'})
    # Rename column 'E-mail' to 'email'
    leads_1 = leads_1.rename(columns={'E-mail': 'email', 'Telefone': 'phone'})
    # Drop columns: 'Versão Página Captura', 'Hora de inscrição' and 5 other columns
    leads_1 = leads_1.drop(columns=['Versão Página Captura', 'Hora de inscrição', 'UTM Source', 'UTM Campaign', 'UTM Medium', 'UTM Content', 'UTM Term'])
    # Convert text to lowercase in column: 'email'
    leads_1['email'] = leads_1['email'].str.lower()
    # Remove leading and trailing whitespace in column: 'email'
    leads_1['email'] = leads_1['email'].str.strip()
    leads_1['new_phone'] = leads_1['phone'].apply(
        lambda x: (
            sanitized if (sanitized := re.sub(r'[^0-9]', '', str(x))) else pd.NA
        )
    )
    # Drop column: 'phone'
    leads_1 = leads_1.drop(columns=['phone'])
    # Rename column 'new_phone' to 'phone'
    leads_1 = leads_1.rename(columns={'new_phone': 'phone'})
    leads_1 = leads_1.dropna(subset=['email', 'phone'])
    return leads_1

leads_1 = clean_data(leads_1.copy())

In [ ]:
leads_2 = pd.read_csv('data/bases_antigas/l09_leads.csv')

import re

def clean_data(leads_2):
    # Drop columns: 'id', 'capture_page_version' and 18 other columns
    leads_2 = leads_2.drop(columns=['id', 'capture_page_version', 'registration_time', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_term', 'utm_content', 'active_id', 'has_taken_course', 'occupation', 'inserted_into_sheet', 'has_entered_group', 'has_left_group', 'date_group_entry', 'date_group_left', 'has_received_onboarding', 'has_anwsered_onboarding', 'has_anwsered_survey', 'survey_response_token'])
    def sanitize_phone(x):
        sanitized = re.sub(r'[^0-9]', '', str(x))
        return sanitized if sanitized else pd.NA
    leads_2['phone'] = leads_2['phone'].apply(sanitize_phone)
    # Convert text to lowercase in column: 'email'
    leads_2['email'] = leads_2['email'].str.lower()
    # Remove leading and trailing whitespace in column: 'email'
    leads_2['email'] = leads_2['email'].str.strip()
    # Drop rows with missing data in columns: 'email', 'phone'
    leads_2 = leads_2.dropna(subset=['email', 'phone'])
    return leads_2

leads_2 = clean_data(leads_2.copy())

In [ ]:
leads_concat = pd.concat([leads_1, leads_2], ignore_index=True)
leads_concat = leads_concat.drop_duplicates(subset=['email'])

In [ ]:
# Realiza uma junção entre survey_concat e leads_concat com base no email,
# mantendo apenas os registros que possuem email.
result = pd.merge(
    leads_concat.dropna(subset=["email"]), 
    survey_concat.dropna(subset=["email"]), 
    on="email", 
    how="inner"
)


In [ ]:
leads_3 = pd.read_csv('data/bases_antigas/le24.csv')
"""
Cell generated by Data Wrangler.
"""
def clean_data(leads_3):
    leads_3 = leads_3.drop(columns=['O preço do Cronograma dos Fluentes, hoje, é de R$ 1.997,00. Caso surja uma vaga e seu perfil seja aprovado, como você gostaria de prosseguir?\n', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_term', 'utm_content', 'Response Type', 'Start Date (UTC)', 'Stage Date (UTC)', 'Submit Date (UTC)', 'Network ID', 'Tags', 'Ending', 'Qual o seu nível de inglês?', '#'])
    leads_3 = leads_3.rename(columns={'Qual o seu nome completo?': 'name', 'Beleza! Agora, qual o seu melhor e-mail?': 'email','Certo! Qual o seu WhatsApp com DDD?': 'phone', 'Qual sua faixa etária?': 'age_range', 'Qual sua renda mensal?': 'incomme'  })
    leads_3 = leads_3.dropna(subset=['incomme'])
    leads_3['email'] = leads_3['email'].str.lower()
    leads_3['email'] = leads_3['email'].str.strip()
    leads_3 = leads_3[(~leads_3['incomme'].str.contains("Não", regex=False, na=False, case=False)) & (~leads_3['incomme'].str.contains("Até", regex=False, na=False, case=False))]
    def sanitize_phone(x):
        sanitized = re.sub(r'[^0-9]', '', str(x))
        return sanitized if sanitized else pd.NA
    leads_3['phone'] = leads_3['phone'].apply(sanitize_phone)
    return leads_3

leads_3 = clean_data(leads_3.copy())

In [ ]:
leads_4 = pd.read_csv('data/bases_antigas/le24-2.csv')
"""
Cell generated by Data Wrangler.
"""
def clean_data(leads_4):
    # Drop columns: '#', 'Qual o seu @ do instagram?' and 24 other columns
    leads_4 = leads_4.drop(columns=['#', 'Qual o seu @ do instagram?', 'Você já fez curso de inglês antes?', 'Qual seu nível atual de conhecimento em inglês?', 'Qual sua ocupação atual?', 'Em qual setor você trabalha?', 'Qual o seu objetivo em relação a fluência em inglês?', 'Me conte com um pouco mais de detalhes seus objetivos. Como você enxerga que o inglês pode mudar sua vida?', 'Qual a sua maior dificuldade hoje para alcançar a fluência em inglês?', 'Me conte com um pouco mais de detalhes, quais dificuldades você enfrenta por não falar inglês fluentemente hoje?', 'Considerando a sua necessidade do idioma e a sua rotina atual, quando pretende iniciar seus estudos no inglês?', 'O preço do Cronograma dos Fluentes, hoje, é de R$ 1.997,00. Como você gostaria de prosseguir?', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_term', 'utm_content', 'counter_05b57750_545a_4790_af5f_f282889a893f', 'Score', 'Start Date (UTC)', 'Response Type', 'Stage Date (UTC)', 'Submit Date (UTC)', 'Network ID', 'Tags', 'Ending'])
    # Rename column 'Qual o seu nome completo?' to 'name'
    leads_4 = leads_4.rename(columns={'Qual o seu nome completo?': 'name'})
    # Rename column 'Beleza! Prazer "te conhecer". Agora, qual o seu melhor e-mail?' to 'email'
    leads_4 = leads_4.rename(columns={'Beleza! Prazer "te conhecer". Agora, qual o seu melhor e-mail?': 'email'})
    # Rename column 'Certo! Qual o seu WhatsApp com DDD?' to 'phone'
    leads_4 = leads_4.rename(columns={'Certo! Qual o seu WhatsApp com DDD?': 'phone'})
    # Rename column 'Qual sua faixa etária?' to 'age_range'
    leads_4 = leads_4.rename(columns={'Qual sua faixa etária?': 'age_range'})
    # Rename column 'Qual a sua renda mensal atual? Mencionar agora, vai te ajudar a reconhecer o quanto o inglês te ajudou a ganhar mais.' to 'incomme'
    leads_4 = leads_4.rename(columns={'Qual a sua renda mensal atual? Mencionar agora, vai te ajudar a reconhecer o quanto o inglês te ajudou a ganhar mais.': 'incomme'})
    leads_4 = leads_4.dropna(subset=['incomme'])
    leads_4['email'] = leads_4['email'].str.lower()
    leads_4['email'] = leads_4['email'].str.strip()
    leads_4 = leads_4[(~leads_4['incomme'].str.contains("Não", regex=False, na=False, case=False)) & (~leads_4['incomme'].str.contains("Até", regex=False, na=False, case=False))]
    def sanitize_phone(x):
        sanitized = re.sub(r'[^0-9]', '', str(x))
        return sanitized if sanitized else pd.NA
    leads_4['phone'] = leads_4['phone'].apply(sanitize_phone)
    return leads_4

leads_4 = clean_data(leads_4.copy())

In [ ]:
final = pd.concat([result, leads_3, leads_4], ignore_index=True)
"""
Cell generated by Data Wrangler.
"""
def clean_data(final):
    # Drop duplicate rows in column: 'phone'
    final = final.drop_duplicates(subset=['phone'])
    # Drop rows with missing data in column: 'email'
    final = final.dropna(subset=['email'])
    return final

final = clean_data(final.copy())

In [ ]:
alunos_tmb = pd.read_csv('data/alunos/alunos_tmb.csv',sep=';')

def clean_data(alunos_tmb):
    # Drop columns: 'Pedido', 'Produtor' and 31 other columns
    alunos_tmb = alunos_tmb.drop(columns=['Pedido', 'Produtor', 'Produto', 'Cliente CPF', 'Endereço completo', 'Logradouro', 'Número', 'Bairro', 'Complemento', 'CEP', 'Cidade', 'Estado', 'País', 'Status', 'Status Financeiro', 'Status Cancelamento', 'Ticket (R$)', 'Modalidade de Contrato', 'Criado Em', 'Data Efetivado', 'Data Cancelado', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_last_source', 'utm_last_medium', 'utm_last_campaign', 'utm_last_content', 'Renovação automatica', 'Data de Renovação Prevista', 'Renovação enviada', 'Oferta'])
    # Drop column: 'Cliente Nome'
    alunos_tmb = alunos_tmb.drop(columns=['Cliente Nome'])
    # Rename column 'Cliente Email' to 'email'
    alunos_tmb = alunos_tmb.rename(columns={'Cliente Email': 'email'})
    # Rename column 'Telefone' to 'phone'
    alunos_tmb = alunos_tmb.rename(columns={'Telefone': 'phone'})
    # Convert text to lowercase in column: 'email'
    alunos_tmb['email'] = alunos_tmb['email'].str.lower()
    # Remove leading and trailing whitespace in columns: 'email', 'phone'
    alunos_tmb['email'] = alunos_tmb['email'].str.strip()
    alunos_tmb['phone'] = alunos_tmb['phone'].str.strip()
    def sanitize_phone(x):
        sanitized = re.sub(r'[^0-9]', '', str(x))
        return sanitized if sanitized else pd.NA
    alunos_tmb['new_phone'] = alunos_tmb['phone'].apply(sanitize_phone)
    return alunos_tmb

alunos_tmb = clean_data(alunos_tmb.copy())


In [ ]:
alunos_hotmart = pd.read_csv('data/alunos/alunos_hotmart.csv')
alunos_generic = pd.read_csv('data/alunos/alunos_generic.csv')

"""
Cell generated by Data Wrangler.
"""
def clean_data(alunos_generic):
    # Convert text to lowercase in column: 'email'
    alunos_generic['email'] = alunos_generic['email'].str.lower()
    # Remove leading and trailing whitespace in columns: 'email', 'phone'
    alunos_generic['email'] = alunos_generic['email'].str.strip()
    alunos_generic['phone'] = alunos_generic['phone'].str.strip()
    def sanitize_phone(x):
        sanitized = re.sub(r'[^0-9]', '', str(x))
        return sanitized if sanitized else pd.NA
    alunos_generic['phone'] = alunos_generic['phone'].apply(sanitize_phone)
    return alunos_generic

alunos_generic = clean_data(alunos_generic.copy())

In [ ]:
bf25_inscritos = pd.read_csv('data/bf25_inscritos/bf25_leads_Inscritos.csv')

"""
Cell generated by Data Wrangler.
"""
def clean_data(bf25_inscritos):
    # Drop columns: 'lead_id', 'name' and 10 other columns
    bf25_inscritos = bf25_inscritos.drop(columns=['lead_id', 'name', 'origem', 'utm_source', 'faixa_de_idade', 'renda', 'ja_fez_outro_curso', 'biggest_fluency_challange', 'principal_objetivo', 'score', 'renda_qualificada', 'qualificado_geral'])
    # Rename column 'formatted_phone' to 'phone'
    bf25_inscritos = bf25_inscritos.rename(columns={'formatted_phone': 'phone'})
    bf25_inscritos['email'] = bf25_inscritos['email'].str.lower()
    # Remove leading and trailing whitespace in columns: 'email', 'phone'
    bf25_inscritos['email'] = bf25_inscritos['email'].str.strip()
    bf25_inscritos['phone'] = bf25_inscritos['phone'].str.strip()
    
    def sanitize_phone(x):
        sanitized = re.sub(r'[^0-9]', '', str(x))
        return sanitized if sanitized else pd.NA
    bf25_inscritos['phone'] = bf25_inscritos['phone'].apply(sanitize_phone)
    return bf25_inscritos

bf25_inscritos = clean_data(bf25_inscritos.copy())


In [ ]:
emails_to_exclude = set()
emails_to_exclude.update(alunos_tmb.email.unique())
emails_to_exclude.update(alunos_hotmart.email.unique())
emails_to_exclude.update(alunos_generic.email.unique())
emails_to_exclude.update(bf25_inscritos.email.unique())

phones_to_exclude = set()
phones_to_exclude.update(alunos_tmb.phone.unique())
phones_to_exclude.update(alunos_generic.phone.unique())
phones_to_exclude.update(bf25_inscritos.phone.unique())

In [ ]:
final = final[~final['email'].isin(emails_to_exclude)]
final = final[~final['phone'].isin(phones_to_exclude)]
final = final.drop_duplicates(subset=['email'])

final['lead_id'] = final['email']